# Prepare h5ad object for label transfer
Using logistic regression

In [2]:
# Load packages
suppressPackageStartupMessages({
    library(reticulate)
    library(BiocParallel)
    library(SingleCellExperiment)
    library(dplyr)
    library(Matrix)
})
    ncores = 4
    mcparam = MulticoreParam(workers = ncores)
    register(mcparam)

options(repr.plot.width=15, repr.plot.height=8)

# define directories
main = "/rds/project/rds-SDzz0CATGms/users/bt392/phd_01_Eomes_RNA/"
in_dir = paste0(main, '07_atlas_mapping/')
out_dir = paste0(main, '07_atlas_mapping/label_transfer/')
dir.create(out_dir, showWarnings = FALSE)


# load in data
big_sce = readRDS(paste0(in_dir,"big_sce.rds"))

chim_meta =  read.csv2(paste0(main, 'meta.tab'), sep='\t')

atlas_in = '/rds/project/rds-SDzz0CATGms/users/bt392/mouse/Mixl1_KO/atlas/atlas/'
atlas_meta = read.table(paste0(atlas_in, 'meta.tab'), header = TRUE, sep = "\t", stringsAsFactors = FALSE, comment.char = "$")

# match formats
chim_meta$cell = paste0('chim_', chim_meta$cell)
atlas_meta$cell = paste0('atlas_', atlas_meta$cell)
chim_meta$sample = paste0('chim_', chim_meta$sample)
atlas_meta$sample = paste0('atlas_', atlas_meta$sample)

atlas_meta = atlas_meta[!atlas_meta$doublet,]
atlas_meta = atlas_meta[!atlas_meta$stripped,]

chim_meta = chim_meta[!chim_meta$doublet,]
chim_meta = chim_meta[!chim_meta$stripped,]

chim_meta = chim_meta %>% select(cell, barcode, stage, sample, Sample_name, tdTom, louvain_cluster)
atlas_meta = atlas_meta %>% select(cell, barcode, stage, sample, celltype, cluster)

# write metadata
write.table(chim_meta, file = paste0(out_dir, "chim_meta.tsv"), sep='\t', col.names = TRUE, row.names = FALSE, quote = FALSE)
write.table(atlas_meta, file = paste0(out_dir, "atlas_meta.tsv"), sep='\t', col.names = TRUE, row.names = FALSE, quote = FALSE)

# separate atlas from experiment
chim_sce = big_sce[, chim_meta$cell]
atlas_sce = big_sce[, atlas_meta$cell]

# write sce
saveRDS(chim_sce, file = paste0(out_dir,"chim_sce.rds"))
saveRDS(atlas_sce, file = paste0(out_dir,"atlas_sce.rds"))